In [2]:
import importlib
import json
import sys
from pathlib import Path


def find_project_dir(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")


PROJECT_DIR = find_project_dir(Path.cwd().resolve())
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import scripts.vpn_switcher as vpn_switcher_module

vpn_switcher = importlib.reload(vpn_switcher_module)
print("프로젝트 경로:", PROJECT_DIR)


프로젝트 경로: /home/wonbeenlee/WorkSpace/boot/SKN28-1st-4team/data_collection


In [3]:
VPN_PRIMARY_TARGETS = ["kr115", "kr142", "kr106", "kr117", "kr153", "kr94", "kr120"]
VPN_FALLBACK_TARGETS = ["jp789", "jp515", "jp429", "jp706", "jp765", "jp570", "jp589", "jp584"]
SWITCH_ATTEMPTS = 5
VPN_PROBE_URL = "https://www.carku.kr/search/car-detail.html?wDemoNo=0361001191"
VPN_CONNECT_TIMEOUT_SECONDS = 45.0
VPN_READY_TIMEOUT_SECONDS = 90.0
VPN_STATUS_POLL_INTERVAL_SECONDS = 2.0
VPN_PROBE_TIMEOUT_SECONDS = 15.0
VPN_MAX_PROBE_LATENCY_SECONDS = 6.0

print(f"테스트 대상 한국 서버: {VPN_PRIMARY_TARGETS}")
print(f"테스트 대상 일본 서버: {VPN_FALLBACK_TARGETS}")
print(f"전환 시도 횟수: {SWITCH_ATTEMPTS}")
print(f"프로브 URL: {VPN_PROBE_URL}")


테스트 대상 국가: ['Japan', 'Singapore', 'Canada', 'Germany', 'Australia']
전환 시도 횟수: 5
프로브 URL: https://www.carku.kr/search/car-detail.html?wDemoNo=0361001191


In [4]:
status_before = await vpn_switcher.get_vpn_status()
public_ip_before = await vpn_switcher.fetch_public_ip()
print(json.dumps(status_before.to_dict(), ensure_ascii=False, indent=2))
print({"public_ip_before": public_ip_before})


{
  "raw_output": "Status: Connected\nServer: Japan #564\nHostname: jp564.nordvpn.com\nIP: 91.207.174.147\nCountry: Japan\nCity: Tokyo\nCurrent technology: NORDLYNX\nCurrent protocol: UDP\nPost-quantum VPN: Disabled\nTransfer: 4.48 GiB received, 134.27 MiB sent\nUptime: 1 hour 38 minutes 30 seconds\n",
  "status": "Connected",
  "server": "Japan #564",
  "hostname": "jp564.nordvpn.com",
  "ip": "91.207.174.147",
  "country": "Japan",
  "city": "Tokyo",
  "uptime": "1 hour 38 minutes 30 seconds",
  "is_connected": true
}
{'public_ip_before': '2.56.252.18'}


In [5]:
switch_results = []
fallback_target_index = 0

for attempt in range(1, SWITCH_ATTEMPTS + 1):
    print(f"===== VPN 전환 테스트 {attempt}/{SWITCH_ATTEMPTS} =====")
    target_sequence = VPN_PRIMARY_TARGETS + VPN_FALLBACK_TARGETS[fallback_target_index:] + VPN_FALLBACK_TARGETS[:fallback_target_index]
    switch_result = await vpn_switcher.rotate_vpn_connection(
        targets=target_sequence,
        start_index=0,
        max_attempts=len(target_sequence),
        connect_timeout_seconds=VPN_CONNECT_TIMEOUT_SECONDS,
        ready_timeout_seconds=VPN_READY_TIMEOUT_SECONDS,
        poll_interval_seconds=VPN_STATUS_POLL_INTERVAL_SECONDS,
        probe_target_url=VPN_PROBE_URL,
        probe_timeout_seconds=VPN_PROBE_TIMEOUT_SECONDS,
        max_probe_latency_seconds=VPN_MAX_PROBE_LATENCY_SECONDS,
    )
    switch_results.append(switch_result)
    switch_result["target_sequence"] = target_sequence
    print(json.dumps(switch_result, ensure_ascii=False, indent=2))

    if not switch_result.get("success"):
        print("전환 실패로 테스트를 중단합니다.")
        break

    target = switch_result.get("target")
    if target in VPN_FALLBACK_TARGETS:
        fallback_target_index = (VPN_FALLBACK_TARGETS.index(target) + 1) % len(VPN_FALLBACK_TARGETS)

switch_results


===== VPN 전환 테스트 1/5 =====
{
  "success": true,
  "target": "Japan",
  "next_target_index": 1,
  "current_ip": "194.233.100.222",
  "error": null,
  "elapsed_seconds": 2.49,
  "attempted_targets": [
    "Japan"
  ],
  "status_before": {
    "raw_output": "Status: Connected\nServer: Japan #564\nHostname: jp564.nordvpn.com\nIP: 91.207.174.147\nCountry: Japan\nCity: Tokyo\nCurrent technology: NORDLYNX\nCurrent protocol: UDP\nPost-quantum VPN: Disabled\nTransfer: 4.48 GiB received, 134.28 MiB sent\nUptime: 1 hour 38 minutes 32 seconds\n",
    "status": "Connected",
    "server": "Japan #564",
    "hostname": "jp564.nordvpn.com",
    "ip": "91.207.174.147",
    "country": "Japan",
    "city": "Tokyo",
    "uptime": "1 hour 38 minutes 32 seconds",
    "is_connected": true
  },
  "status_after": {
    "raw_output": "Status: Connected\nServer: Japan #782\nHostname: jp782.nordvpn.com\nIP: 194.233.100.222\nCountry: Japan\nCity: Tokyo\nCurrent technology: NORDLYNX\nCurrent protocol: UDP\nPost-qua

[{'success': True,
  'target': 'Japan',
  'next_target_index': 1,
  'current_ip': '194.233.100.222',
  'error': None,
  'elapsed_seconds': 2.49,
  'attempted_targets': ['Japan'],
  'status_before': {'raw_output': 'Status: Connected\nServer: Japan #564\nHostname: jp564.nordvpn.com\nIP: 91.207.174.147\nCountry: Japan\nCity: Tokyo\nCurrent technology: NORDLYNX\nCurrent protocol: UDP\nPost-quantum VPN: Disabled\nTransfer: 4.48 GiB received, 134.28 MiB sent\nUptime: 1 hour 38 minutes 32 seconds\n',
   'status': 'Connected',
   'server': 'Japan #564',
   'hostname': 'jp564.nordvpn.com',
   'ip': '91.207.174.147',
   'country': 'Japan',
   'city': 'Tokyo',
   'uptime': '1 hour 38 minutes 32 seconds',
   'is_connected': True},
  'status_after': {'raw_output': 'Status: Connected\nServer: Japan #782\nHostname: jp782.nordvpn.com\nIP: 194.233.100.222\nCountry: Japan\nCity: Tokyo\nCurrent technology: NORDLYNX\nCurrent protocol: UDP\nPost-quantum VPN: Disabled\nTransfer: 2.61 KiB received, 1.16 KiB 

In [6]:
status_after = await vpn_switcher.get_vpn_status()
public_ip_after = await vpn_switcher.fetch_public_ip()
print(json.dumps(status_after.to_dict(), ensure_ascii=False, indent=2))
print({"public_ip_after": public_ip_after})

summary = {
    "attempt_count": len(switch_results),
    "success_count": sum(1 for item in switch_results if item.get("success")),
    "failure_count": sum(1 for item in switch_results if not item.get("success")),
    "attempted_targets": [item.get("target") for item in switch_results],
    "final_fallback_target_index": fallback_target_index,
}
print(json.dumps(summary, ensure_ascii=False, indent=2))


{
  "raw_output": "Status: Disconnected\n",
  "status": "Disconnected",
  "server": null,
  "hostname": null,
  "ip": null,
  "country": null,
  "city": null,
  "uptime": null,
  "is_connected": false
}
{'public_ip_after': '220.121.38.51'}
{
  "attempt_count": 4,
  "success_count": 3,
  "failure_count": 1,
  "attempted_targets": [
    "Japan",
    "Japan",
    "Australia",
    null
  ]
}
